In [ ]:
from cil.framework import AcquisitionGeometry
from cil.optimisation.algorithms import ISTA, FISTA
from cil.optimisation.functions import SVRGFunction, LSVRGFunction, LeastSquares
from cil.optimisation.utilities import RandomSampling, MetricsDiagnostics
from cil.plugins.astra import FBP, ProjectionOperator
from TotalVariation import TotalVariationNew
from ProxSkip import ProxSkip
from utils import create_circular_mask, list_of_functions, StoppingCriterion

import numpy as np
import zarr
import matplotlib.pyplot as plt

plt.rcParams['lines.linewidth'] = 5
plt.rcParams['lines.markersize'] = 20
plt.rcParams['font.size'] = 40
plt.rcParams["image.cmap"]  = "inferno"

### Total Variation Reconstruction on a real dataset

In this notebook, we play the following game. We start from a high-accuracy reference solution of

$$ 
\begin{equation}
\min_{x\in\mathbb{R}^n} \; \frac{1}{2}\|Ax - b\|_{2}^{2}
+ \alpha \mathrm{TV}(x)
+ \mathbb{I}_{\{x \ge 0\}}(x),
\end{equation}
$$

and we fix a target accuracy threshold $\varepsilon>0$. The goal is to compare algorithms by **time-to-accuracy**: namely, determine which algorithm reaches $\frac{\|x_k - x^*\|_2}{\|x^*\|_2}$ below $\varepsilon$ in the **shortest runtime**.


### Load real dataset

In [ ]:
sino = zarr.load("data/NiPd_spent_01_microct_rings_removed_2D.zarr")

print(f"sinogram shape is {sino.shape}")

_, horizontal = sino.shape

angles_list = np.linspace(0, np.pi, 800)[::2]
ag2D = AcquisitionGeometry.create_Parallel2D().\
        set_panel(horizontal).\
        set_angles(angles_list, angle_unit="radian").\
        set_labels(['angle','horizontal'])
ig2D = ag2D.get_ImageGeometry()

data2D = ag2D.allocate()
data2D.fill(sino)

### FBP reconstruction

In [ ]:
fbp = FBP(ig2D, ag2D, device="cpu")(data2D)
fbp.array[fbp.array<0] = 0
plt.imshow(fbp.array)

### Load high accuracy solution 

In [ ]:
alpha = 0.1

In [ ]:
pdhg_optimal_info = zarr.open_group("data/pdhg_optimal_finden_tv_alpha_{}_explicit_precond_maxiterations_200000.zarr".format(alpha))

pdhg_optimal_np = pdhg_optimal_info["solution"][:]
pdhg_optimal_cil = ig2D.allocate()
pdhg_optimal_cil.fill(pdhg_optimal_np)

### Family of considered algorithms (ProxSkip template)

**Parameters:** $\gamma>0$, probability $p\in(0,1]$, data subsets $N$
**Initialize:** $x_0, h_0 \in \mathbb{R}^n$

For $k=0,1,\dots,K-1$:

1. Compute $G_k$ (an unbiased estimator of $\nabla f(x_k)$).
2. $$\hat x_{k+1} = x_k - \gamma\big(G_k(x_k) - h_k\big).$$
3. Sample $\theta_k\sim\mathrm{Bernoulli}(p)$, $\theta_k\in{0,1}$.
4. If $\theta_k=1$:
   $$x_{k+1}=\mathrm{prox}_{\frac{\gamma}{p}g}\left(\hat x_{k+1}-\frac{\gamma}{p}h_k\right),$$
   else:
   $$x_{k+1}=\hat x_{k+1}.$$
5. Update:
   $$h_{k+1}=h_k+\frac{p}{\gamma}\big(x_{k+1}-\hat x_{k+1}\big).$$

---

| $p=1$     | $0<p<1$       | $G_k$                                                                |
| --------- | ------------- | -------------------------------------------------------------------- |
| ISTA      | ProxSkip      | $\nabla f(x_k)$                                                      |
| ProxSGD   | ProxSGDSkip   | $N\nabla f_{i_k}(x_k)$                                               |
| ProxSAGA  | ProxSAGASkip  | $N(\nabla f_{i_k}(x_k)-v_k^{,i_k})+\bar v_k$                         |
| ProxSVRG  | ProxSVRGSkip  | $N(\nabla f_{i_k}(x_k)-\nabla f_{i_k}(\tilde x))+\nabla f(\tilde x)$ |
| ProxLSVRG | ProxLSVRGSkip | as above, updated with $p=1/N$                                       |

*Note:* FISTA is ISTA with an acceleration step (Beck & Teboulle).


### Define Stopping Criteria and Metrics (PSNR/SSIM)

In [ ]:
h, w = pdhg_optimal_cil.shape
mask = create_circular_mask(h, w, center=(170,165), radius=150)

def NRSE(x, y, **kwargs):
    return np.sqrt(np.sum(np.abs(x - y*mask.ravel())**2))/np.sqrt(np.sum(x**2))

cb_metrics = MetricsDiagnostics(reference_image=pdhg_optimal_cil*mask, 
                         metrics_dict={"rse":NRSE}) 

epsilon = 0.99e-5
inner_its = 10
epochs = 100
max_iteration = 5000
seed_skip = 42
seed_sampling = 42

In [ ]:
### Avoid computing objectives, not needed for this demo
def update_objective(self):
    return 0.

ISTA.update_objective = update_objective
ProxSkip.update_objective = update_objective
FISTA.update_objective = update_objective

### Run deterministic algorithms: No data splitting, no skipping

In [ ]:
K = ProjectionOperator(ig2D, ag2D, device="cpu")
F = LeastSquares(A=K, b=data2D, c=0.5)
G = alpha * TotalVariationNew(max_iteration = inner_its, tolerance=None, correlation='Space',
                                     backend='c', lower=0, upper=np.infty, isotropic=True, 
                                     split=False, info=False, strong_convexity_constant=0,
                                     warm_start=True)
initial = ig2D.allocate()

### FISTA

In [ ]:
step_size = 1./F.L
cb_error = StoppingCriterion(epsilon=epsilon) 
fista = FISTA(initial = initial, f = F, step_size = step_size, g=G, 
        update_objective_interval = 1,
        max_iteration = max_iteration) 
fista.run(verbose=0, callback=[cb_metrics, cb_error])

### ISTA

In [ ]:
step_size = 1.99/F.L
cb_error = StoppingCriterion(epsilon=epsilon) 
ista = ISTA(initial = initial, f = F, step_size = step_size, g=G, 
        update_objective_interval = 1,
        max_iteration = max_iteration) 
ista.run(verbose=0, callback=[cb_metrics, cb_error])

### Run ProxSkip: Skip the regulariser

In [ ]:
prob = 0.1

In [ ]:
step_size = 1.99/F.L
cb_error = StoppingCriterion(epsilon=epsilon) 
proxskip = ProxSkip(initial = [initial, initial], f = F, step_size = step_size, g=G, 
        update_objective_interval = 1, prob=prob, seed = seed_skip,
        max_iteration = max_iteration) 
proxskip.run(verbose=0, callback=[cb_metrics, cb_error])


### Run stochastic algorithms: Data splitting, no skipping

In [ ]:
def list_of_functions(data):
    
    list_funcs = []
    ig = data[0].geometry.get_ImageGeometry()
    
    for d in data:
        ageom_subset = d.geometry        
        Ai = ProjectionOperator(ig, ageom_subset, device = 'cpu')    
        fi = LeastSquares(Ai, b = d, c = 0.5)
        list_funcs.append(fi)   
        
    return list_funcs

In [ ]:
nsub = 200
data_split, method = data2D.split_to_subsets(nsub, method= "ordered", info=True)
list_func = list_of_functions(data_split) 

In [ ]:
selection = RandomSampling(len(list_func), nsub, seed=seed_sampling)
Fsvrg = SVRGFunction(list_func, selection = selection, update_frequency=len(list_func))
Fsvrg.initial = initial
step_size = 1./(Fsvrg.L)
cb_error = StoppingCriterion(epsilon=epsilon, epochs = 100) 
prox_svrg = ISTA(initial = initial, f = Fsvrg, step_size = step_size, g=G, 
            update_objective_interval = 1, 
            max_iteration = max_iteration) 
prox_svrg.run(verbose=0, callback=[cb_metrics, cb_error])

In [ ]:
selection = RandomSampling(len(list_func), nsub, seed=seed_sampling)
Flsvrg = LSVRGFunction(list_func, selection = selection, update_prob=1./len(list_func))
Flsvrg.initial = initial
step_size = 1./(Fsvrg.L)
cb_error = StoppingCriterion(epsilon=epsilon, epochs = 100) 
prox_lsvrg = ISTA(initial = initial, f = Flsvrg, step_size = step_size, g=G, 
            update_objective_interval = 1, 
            max_iteration = max_iteration) 
prox_lsvrg.run(verbose=0, callback=[cb_metrics, cb_error])

### Run stochastic algorithms: Data splitting and skipping

In [ ]:
selection = RandomSampling(len(list_func), nsub, seed=seed_sampling)
Fsvrg_skip = SVRGFunction(list_func, selection = selection, update_frequency=len(list_func))
Fsvrg_skip.initial = initial
step_size = 1./(Fsvrg_skip.L)
cb_error = StoppingCriterion(epsilon=epsilon, epochs = 100) 
prox_svrg_skip = ProxSkip(initial = [initial, initial], f = Fsvrg_skip, step_size = step_size, g=G, 
            update_objective_interval = 1, prob = prob,
            max_iteration = max_iteration) 
prox_svrg_skip.run(verbose=0, callback=[cb_metrics, cb_error])

In [ ]:
selection = RandomSampling(len(list_func), nsub, seed=seed_sampling)
Flsvrg_skip = LSVRGFunction(list_func, selection = selection, update_prob=1./len(list_func))
Flsvrg_skip.initial = initial
step_size = 1./(Flsvrg_skip.L)
cb_error = StoppingCriterion(epsilon=epsilon, epochs = 100) 
prox_lsvrg_skip = ProxSkip(initial = [initial, initial], f = Flsvrg_skip, step_size = step_size, g=G, 
            update_objective_interval = 1, prob = prob,
            max_iteration = max_iteration) 
prox_lsvrg_skip.run(verbose=0, callback=[cb_metrics, cb_error])

### Plot PSNR/SSIM progress
- with respect to iteration
- with respect to time

In [ ]:
def find_iter_to_error(algo, error):
    rse = np.asarray(algo.rse)
    idx = np.where(rse < error)[0]
    return np.nan if idx.size == 0 else int(idx[0])

def time_to_iter(algo, it):
    """Time up to and including iteration it."""
    if it is None or (isinstance(it, float) and np.isnan(it)):
        return np.nan
    it = int(it)
    return float(np.sum(np.asarray(algo.timing)[:it+1]))

def time_to_error(algo, error):
    it = find_iter_to_error(algo, error)
    return time_to_iter(algo, it)

In [ ]:
t_ista = time_to_error(ista, epsilon)
t_proxskip = time_to_error(proxskip, epsilon)
t_prox_svrg =time_to_error(prox_svrg, epsilon)
t_prox_lsvrg = time_to_error(prox_lsvrg, epsilon)
t_prox_svrg_skip = time_to_error(prox_svrg_skip, epsilon)
t_prox_lsvrg_skip = time_to_error(prox_lsvrg_skip, epsilon)

In [ ]:
fig, ax = plt.subplots(2,1,figsize=(30, 25))

fig.subplots_adjust(top=0.80)

ax[0].semilogy(prox_svrg.rse[:-1],label=f"ProxSVRG (N=200)")
ax[0].semilogy(prox_lsvrg.rse[:-1],label=f"ProxLSVRG (N=200)")
ax[0].semilogy(proxskip.rse[:-1],label=f"ProxSkip (p=0.1)")
ax[0].semilogy(prox_svrg_skip.rse[:-1],label=f"ProxSVRGSkip (N=200, p=0.05)")
ax[0].semilogy(prox_lsvrg_skip.rse[:-1],label=f"ProxLSVRGSkip (N=200, p=0.05)")
ax[0].semilogy(ista.rse[:-1],label=f"ISTA")
ax[0].semilogy(fista.rse[:-1],label=f"FISTA")
ax[0].set_xlabel("Iteration")
ax[0].set_ylabel(r"$\frac{\|x_{k} - x^{*}\|_{2}}{\|x^{*}\|_{2}}$")
ax[0].grid(True, which="major")

ax[0].text(
    0.25, 0.9, "Proximal-TV with 10 iterations",
    transform=ax[0].transAxes,
    ha="left", va="top",
    fontsize=45,
    bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="none")
)
ax[0].legend(ncols=2, loc="lower center", bbox_to_anchor=(0.5, 1.00), frameon=True)

ax[1].semilogy(np.cumsum(prox_svrg.timing), prox_svrg.rse[:-1],label=f"ProxSVRG (N=200)")
ax[1].semilogy(np.cumsum(prox_lsvrg.timing), prox_lsvrg.rse[:-1],label=f"ProxLSVRG (N=200)")
ax[1].semilogy(np.cumsum(proxskip.timing), proxskip.rse[:-1],label=f"ProxSkip (p=0.1)")
ax[1].semilogy(np.cumsum(prox_svrg_skip.timing), prox_svrg_skip.rse[:-1],label=f"ProxSVRGSkip (N=200, p=0.05)")
ax[1].semilogy(np.cumsum(prox_lsvrg_skip.timing), prox_lsvrg_skip.rse[:-1],label=f"ProxLSVRGSkip (N=200, p=0.05)")
ax[1].semilogy(np.cumsum(ista.timing), ista.rse[:-1],label=f"ISTA")
ax[1].semilogy(np.cumsum(fista.timing), fista.rse[:-1],label=f"FISTA")
ax[1].set_xlabel("Time (sec)")
ax[1].set_ylabel(r"$\frac{\|x_{k} - x^{*}\|_{2}}{\|x^{*}\|_{2}}$")
ax[1].grid(True, which="major")

plt.show()